# ADSP 31014 — Statistical Models for Data Science
## Homework 3

**Name:** Obaro Erigha    
**Term:** Autumn 2026    

In [38]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

df = pd.read_csv('C:\\Users\\Obaro\\UChicagoProjects\\Statistical Models for Data Science\\Week2\\Lecture2\\chicago_taxi_trip_sample.csv')
print(df.shape)
df.head()

(100000, 17)


,trip_id,taxi_id,month,day_of_week,hour,pickup_area,dropoff_area,trip_miles,trip_seconds,fare,tips,tips_percent,tolls,extras,trip_total,payment_type,company
0,259f9c40ef9,f9bc93,1,5,0,Other,Other,0.35,151,4.25,0.00,0.000,0.0,1.0,5.25,Cash,City Service
1,0f61207c6d8,271cde,1,5,0,Downtown,Other,0.53,918,8.50,0.00,0.000,0.0,0.0,8.50,Cash,Sun Taxi
2,252b55a174d,a37d54,1,5,0,Downtown,Downtown,1.05,439,9.36,2.14,0.229,0.0,0.0,11.50,Other,Medallion Leasin
3,b2b5c1ccfdd,3c0868,1,5,0,Downtown,Other,6.73,1277,19.75,5.44,0.275,0.0,1.5,27.19,Credit Card,Sun Taxi
4,5edc7fe28fc,d9587f,1,5,0,Downtown,Downtown,0.90,540,6.50,2.00,0.308,0.0,1.0,10.00,Credit Card,Globe Taxi


In [50]:
print(df[['trip_seconds', 'fare', 'hour', 'pickup_area']].info())
print(df['hour'].value_counts(), '\n')
print(df['pickup_area'].value_counts())

df = df.dropna(subset=['trip_seconds', 'fare', 'hour', 'pickup_area']).reset_index(drop=True)

print(df.shape)
print(df[['trip_seconds', 'fare', 'hour', 'pickup_area']].info())
print(df['hour'].value_counts(), '\n')
print(df['pickup_area'].value_counts())

area_levels = sorted(df['pickup_area'].unique())
print('\npickup_area levels:', area_levels)

hour_type = {'Rush Hour': {7, 8, 9, 15, 16, 17, 18},
             'Night': {0, 1, 2, 3, 4, 5, 6, 21, 22, 23},
             'Other': {10, 11, 12, 13, 14, 19, 20}}

OHARE = [a for a in area_levels if 'hare' in a.lower()][0]
print("O'Hare label in data:", repr(OHARE))

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   trip_seconds  100000 non-null  int64  
 1   fare          100000 non-null  float64
 2   hour          100000 non-null  int64  
 3   pickup_area   100000 non-null  str    
dtypes: float64(1), int64(2), str(1)
memory usage: 3.7 MB
None
hour
17    7214
16    7016
18    6953
15    6786
14    6752
13    6625
12    6371
19    5973
11    5867
10    5655
9     5148
20    4800
8     4592
21    3915
22    3253
7     3064
23    2465
0     1780
6     1742
1     1240
5      906
2      749
4      582
3      552
Name: count, dtype: int64 

pickup_area
Downtown    48862
Other       33391
O'Hare      14725
Midway       3022
Name: count, dtype: int64
(100000, 17)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------  

---
## Question 1a — Two categorical inputs (no interaction)

In [40]:
def equation(params, response='trip_seconds'):
    terms = []

    for name, b in params.items():
        if name == 'Intercept':
            terms.append(f'{b:.4f}')
            continue

        factors = []
        for part in name.split(':'):
            if '[T.' in part:
                var, lvl = part.split('[T.')
                lvl = lvl.rstrip(']')
                if var.startswith('C(') and var.endswith(')'):
                    var = var[2:-1]
                factors.append(f'1{{{var}={lvl}}}')
            else:
                factors.append(part)

        terms.append(f'{b:+.4f}*' + '*'.join(factors))

    return f'{response} = ' + ' '.join(terms)

In [41]:
m1a = smf.ols('trip_seconds ~ hour + pickup_area', data=df).fit()
print(m1a.summary())

                            OLS Regression Results                            
Dep. Variable:           trip_seconds   R-squared:                       0.263
Model:                            OLS   Adj. R-squared:                  0.263
Method:                 Least Squares   F-statistic:                     8913.
Date:                Fri, 23 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:04:39   Log-Likelihood:            -8.0745e+05
No. Observations:              100000   AIC:                         1.615e+06
Df Residuals:                   99995   BIC:                         1.615e+06
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept               800.53

In [42]:
new = pd.DataFrame({'hour': [8], 'pickup_area': ['Downtown']})
print('\nPredicted trip_seconds (hour=8, Downtown):',m1a.predict(new).iloc[0])

print('R^2:', round(m1a.rsquared, 4))

print('Number of beta params:', len(m1a.params))

print(m1a.params.round(4))

print(equation(m1a.params))


Predicted trip_seconds (hour=8, Downtown): 805.9066111436372
R^2: 0.2628
Number of beta params: 5
Intercept                 800.5388
pickup_area[T.Midway]     863.5855
pickup_area[T.O'Hare]    1349.8365
pickup_area[T.Other]      364.7031
hour                        0.6710
dtype: float64
trip_seconds = 800.5388 +863.5855*1{pickup_area=Midway} +1349.8365*1{pickup_area=O'Hare} +364.7031*1{pickup_area=Other} +0.6710*hour


---
## Question 1b — Categorical interaction

In [43]:
m1b = smf.ols('trip_seconds ~ C(hour) * pickup_area', data=df).fit()
RUSH_HOUR = 8

print(m1b.summary())
print('R^2                 :', round(m1b.rsquared, 4))
print('Number of beta params:', len(m1b.params))

new = pd.DataFrame({'hour': [8],'pickup_area': [OHARE]})

print("Predicted trip_seconds (hour placeholder, O'Hare):",
      m1b.predict(new).iloc[0])
grp = (
    df.groupby(['hour', 'pickup_area'])['trip_seconds']
      .agg(['mean', 'count'])
      .reset_index()
)
print('\nGroup means:')
print(grp)

print('\nCross-tab of means:')
print(
    df.pivot_table(
        index='hour',
        columns='pickup_area',
        values='trip_seconds',
        aggfunc='mean'
    ).round(2)
)

imin = grp['mean'].idxmin()
print('\nSmallest sample mean:')
print(grp.loc[imin])

                            OLS Regression Results                            
Dep. Variable:           trip_seconds   R-squared:                       0.326
Model:                            OLS   Adj. R-squared:                  0.325
Method:                 Least Squares   F-statistic:                     508.1
Date:                Fri, 23 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:04:40   Log-Likelihood:            -8.0299e+05
No. Observations:              100000   AIC:                         1.606e+06
Df Residuals:                   99904   BIC:                         1.607e+06
Df Model:                          95                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [44]:
cells = grp[['hour', 'pickup_area']].copy()
cells['fitted'] = m1b.predict(cells)

cmp = cells.merge(grp, on=['hour', 'pickup_area'])
cmp['abs_diff'] = (cmp['fitted'] - cmp['mean']).abs()

print(cmp)
print('Max (fitted - sample mean):', cmp['abs_diff'].max())

    hour pickup_area       fitted         mean  count      abs_diff
0      0    Downtown   651.778287   651.778287    654  1.830358e-11
1      0      Midway  1212.858407  1212.858407    113  2.319211e-11
2      0      O'Hare  1416.650000  1416.650000    460  2.955858e-11
3      0       Other   807.412297   807.412297    553  1.409717e-11
4      1    Downtown   610.478747   610.478747    447  1.716671e-11
..   ...         ...          ...          ...    ...           ...
91    22       Other   927.454545   927.454545   1023  5.456968e-12
92    23    Downtown   672.378351   672.378351    970  4.774847e-12
93    23      Midway  1344.633333  1344.633333    180  3.501555e-11
94    23      O'Hare  1393.083164  1393.083164    493  3.683454e-11
95    23       Other   821.667883   821.667883    822  3.751666e-12

[96 rows x 6 columns]
Max (fitted - sample mean): 2.7125679480377585e-10


In [45]:
def contributes_to_cell(param_name, target):
    """
    True if a patsy coefficient contributes to the target cell.
    target is like {'hour': 8, 'pickup_area': "O'Hare"}.
    """
    for part in param_name.split(':'):
        if '[T.' in part:
            var, lvl = part.split('[T.')
            lvl = lvl.rstrip(']')
            if var.startswith('C(') and var.endswith(')'):
                var = var[2:-1]
            if str(target.get(var)) != lvl:
                return False
    return True

p = m1b.params
target = {'hour': 8, 'pickup_area': OHARE}

names = [n for n in p.index if contributes_to_cell(n, target)]

print('Sum =', p[names].sum())

Sum = 2612.7189189189266


---
## Question 1c — Include `fare`

In [46]:
f1c = 'trip_seconds ~ C(hour) * pickup_area + fare:(C(hour) * pickup_area)'
m1c = smf.ols(f1c, data=df).fit()

print(m1c.summary())
print('R^2:', round(m1c.rsquared, 4))
print('Number of beta params:', len(m1c.params))

                            OLS Regression Results                            
Dep. Variable:           trip_seconds   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     1629.
Date:                Fri, 23 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:04:42   Log-Likelihood:            -7.5193e+05
No. Observations:              100000   AIC:                         1.504e+06
Df Residuals:                   99808   BIC:                         1.506e+06
Df Model:                         191                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

In [47]:
p = m1c.params
target = {'hour': RUSH_HOUR, 'pickup_area': OHARE}

a0_names = [
    n for n in p.index
    if contributes_to_cell(n, target) and 'fare' not in n
]
a1_names = [
    n for n in p.index
    if contributes_to_cell(n, target) and 'fare' in n
]

print('\nIntercept terms used:', a0_names)
print('Slope terms used    :', a1_names)

a0_manual = p[a0_names].sum()
a1_manual = p[a1_names].sum()

print(f'a0 (by summing coefficients) = {a0_manual:.4f}')
print(f'a1 (by summing coefficients) = {a1_manual:.4f}')

nd = pd.DataFrame({
    'hour': [RUSH_HOUR, RUSH_HOUR],
    'pickup_area': [OHARE, OHARE],
    'fare': [0.0, 1.0]
})
yhat = m1c.predict(nd).values

print(f'\na0 (prediction at fare=0) = {yhat[0]:.4f}')
print(f'a1 (pred(1) - pred(0))    = {yhat[1] - yhat[0]:.4f}')


Intercept terms used: ['Intercept', 'C(hour)[T.8]', "pickup_area[T.O'Hare]", "C(hour)[T.8]:pickup_area[T.O'Hare]"]
Slope terms used    : ['fare:C(hour)[0]', 'fare:C(hour)[1]', 'fare:C(hour)[2]', 'fare:C(hour)[3]', 'fare:C(hour)[4]', 'fare:C(hour)[5]', 'fare:C(hour)[6]', 'fare:C(hour)[7]', 'fare:C(hour)[8]', 'fare:C(hour)[9]', 'fare:C(hour)[10]', 'fare:C(hour)[11]', 'fare:C(hour)[12]', 'fare:C(hour)[13]', 'fare:C(hour)[14]', 'fare:C(hour)[15]', 'fare:C(hour)[16]', 'fare:C(hour)[17]', 'fare:C(hour)[18]', 'fare:C(hour)[19]', 'fare:C(hour)[20]', 'fare:C(hour)[21]', 'fare:C(hour)[22]', 'fare:C(hour)[23]', "fare:pickup_area[T.O'Hare]", "fare:C(hour)[T.8]:pickup_area[T.O'Hare]"]
a0 (by summing coefficients) = -82.5977
a1 (by summing coefficients) = 1119.5249

a0 (prediction at fare=0) = -82.5977
a1 (pred(1) - pred(0))    = 64.8018


In [48]:
sub = df[(df['hour'] == RUSH_HOUR) & (df['pickup_area'] == OHARE)]
m_sub = smf.ols('trip_seconds ~ fare', data=sub).fit()

print('n rows in subset:', len(sub))
print(m_sub.summary().tables[1])
print(f'\nSubset fit: a0 = {m_sub.params["Intercept"]:.4f}, '
      f'a1 = {m_sub.params["fare"]:.4f}')

n rows in subset: 370
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -82.5977    197.169     -0.419      0.676    -470.316     305.121
fare          64.8018      4.574     14.167      0.000      55.807      73.796

Subset fit: a0 = -82.5977, a1 = 64.8018


---
## Question 1d — Three-way interaction

In [49]:
m1d = smf.ols('trip_seconds ~ C(hour) * pickup_area * fare',data=df).fit()

print(m1d.summary())
print('R^2 1d:', round(m1d.rsquared, 6))
print('R^2 1c:', round(m1c.rsquared, 6))
print('Number of beta params 1d:', len(m1d.params))
print('Number of beta params 1c:', len(m1c.params))

print('Same fitted values?', np.allclose(m1d.fittedvalues, m1c.fittedvalues))
print('Same residual SS?  ', np.isclose(m1d.ssr, m1c.ssr))

def canon(name):
    return ':'.join(sorted(name.split(':')))

c1 = m1c.params.rename(index=canon).sort_index()
c2 = m1d.params.rename(index=canon).sort_index()
c1, c2 = c1.align(c2)

print('Same coefficients matched by term?',
      np.allclose(c1.values, c2.values))

                            OLS Regression Results                            
Dep. Variable:           trip_seconds   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     1629.
Date:                Fri, 23 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:04:44   Log-Likelihood:            -7.5193e+05
No. Observations:              100000   AIC:                         1.504e+06
Df Residuals:                   99808   BIC:                         1.506e+06
Df Model:                         191                                         
Covariance Type:            nonrobust                                         
                                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

---
## Question 2 — Asking GenAI about categorical variables with many levels



**Prompt used:** "How should I handle a categorical variable with many levels when fitting a linear regression model?"

1. **Why it is a problem.** A categorical variable with K levels needs K−1 dummy variables. With many levels this inflates the number of parameters, so the model overfits, standard errors become large, some levels have very few observations (unstable or unestimable coefficients), and new/unseen levels at prediction time cannot be handled.
2. **Collapse/group levels.** Merge rare levels into an "Other" category, or group levels using domain knowledge.
3. **Use a numeric summary of the level (encoding).** Replace the category with something meaningful such as frequency/count encoding or target (mean) encoding. Target encoding must be computed within cross-validation folds to avoid leakage.
4. **Regularization.** Fit ridge/lasso/elastic net on the dummy variables so that coefficients of weakly supported levels are shrunk (lasso can set them to zero).
5. **Mixed-effects / hierarchical models.** Treat the variable as a random effect (partial pooling), so levels with little data borrow strength from the overall mean. This also handles unseen levels more gracefully.
6. **Feature selection / testing.** Use an F-test (ANOVA) for the variable as a whole instead of looking at individual dummy t-tests, and use cross-validation to choose the amount of grouping or shrinkage.
7. **Practical tips.** Choose a reference level with plenty of data, make sure each level has enough observations, and avoid high-order interactions between several high-cardinality variables, since the parameter count multiplies